# Study and encode record labels

We examine the prepared `record_label` column and document the measured contribution of a group-aware target-encoded feature. We compare only the old model and the same model with this one additional feature. Notebooks 08 and delivery fit only the retained configuration.

We read the prepared CSV files without changing them. We keep target encoding inside training because it uses `music_hit`; a column encoded using all target labels before validation would leak information. Reports are saved to `reports/record_label_encoding/`.

## 1. Configuration and prepared inputs

In [1]:
from pathlib import Path
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from scipy.sparse import csr_matrix, hstack
from IPython.display import display, Image
from threadpoolctl import threadpool_limits
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score,
                             RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, TargetEncoder

RANDOM_STATE = 0
THRESHOLD = 0.5
MAX_ITER = 1000
LABEL_FOLDS = 5
PROCESSED_DIR = next((directory for directory in (
    Path.cwd() / "data/processed", Path.cwd().parent / "data/processed", Path.cwd(),
) if all((directory / name).is_file() for name in ("train.csv", "test.csv"))), None)
if PROCESSED_DIR is None:
    raise FileNotFoundError("Set PROCESSED_DIR to the folder containing the prepared train.csv and test.csv.")
PROJECT_DIR = PROCESSED_DIR.parent.parent if PROCESSED_DIR.name == "processed" else Path.cwd()
REPORT_DIR = PROJECT_DIR / "reports/record_label_encoding"
MODEL_DIR = PROJECT_DIR / "models"
DELIVERY_DIR = PROJECT_DIR / "delivery"
for directory in (REPORT_DIR, MODEL_DIR, DELIVERY_DIR):
    directory.mkdir(parents=True, exist_ok=True)


/home/goncalotorcato/.config/matplotlib is not a writable directory
Matplotlib created a temporary cache directory at /tmp/matplotlib-_q5o1pig because there was an issue with the default path (/home/goncalotorcato/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


In [2]:
train_header = pd.read_csv(PROCESSED_DIR / "train.csv", nrows=0).columns.tolist()
test_header = pd.read_csv(PROCESSED_DIR / "test.csv", nrows=0).columns.tolist()
GENRE_COLUMNS = [column for column in train_header if column.startswith("genre_")]
assert "music_hit" in train_header and "music_hit" not in test_header
assert [column for column in train_header if column != "music_hit"] == test_header
assert "release_year" in train_header and "release_date" not in train_header
assert "artist_followers_log1p" in train_header and "genre_unknown" in GENRE_COLUMNS


def load_prepared_csv(path, columns):
    # Read genre indicators in chunks and keep them sparse to limit memory use.
    parts = []
    for chunk in pd.read_csv(path, chunksize=10000, dtype={column: "uint8" for column in GENRE_COLUMNS}):
        values = chunk[GENRE_COLUMNS].to_numpy(dtype=np.uint8)
        assert ((values == 0) | (values == 1)).all()
        indicators = pd.DataFrame.sparse.from_spmatrix(
            csr_matrix(values), index=chunk.index, columns=GENRE_COLUMNS,
        )
        parts.append(pd.concat([chunk.drop(columns=GENRE_COLUMNS), indicators], axis=1)[columns])
    return pd.concat(parts, ignore_index=True)


model_train = load_prepared_csv(PROCESSED_DIR / "train.csv", train_header)
model_test = load_prepared_csv(PROCESSED_DIR / "test.csv", test_header)
assert model_train["id"].is_unique and model_test["id"].is_unique
assert not set(model_train["id"]).intersection(model_test["id"])
assert set(model_train["music_hit"].unique()) == {0, 1}
assert not model_train["duration_sec"].eq(0).any()
display(pd.DataFrame({
    "dataset": ["prepared train", "prepared test"],
    "rows": [len(model_train), len(model_test)],
    "columns": [model_train.shape[1], model_test.shape[1]],
}))


,dataset,rows,columns
0,prepared train,350049,1025
1,prepared test,87513,1024


## 2. Study the label categories

In [3]:
label_rows = []
for name, frame in (("train", model_train), ("test", model_test)):
    counts = frame["record_label"].value_counts(dropna=False)
    label_rows.append({
        "dataset": name, "distinct record labels": frame["record_label"].nunique(),
        "missing values": int(frame["record_label"].isna().sum()),
        "unknown values": int(frame["record_label"].eq("unknown").sum()),
        "labels with one track": int(counts.eq(1).sum()),
    })
label_summary = pd.DataFrame(label_rows)
display(label_summary)
display(model_train["record_label"].value_counts().head(10).rename("tracks").to_frame())


,dataset,distinct record labels,missing values,unknown values,labels with one track
0,train,31559,0,38,14006
1,test,17781,0,14,7563


,tracks
record_label,
CC Entertainment,4521
Sami Springs,2712
Columbia,2238
Legacy Recordings,2189
Warner Records,1863
RCA Records Label,1812
UME - Global Clearing House,1770
Ameritz Music Ltd,1750
Independent,1644


## 3. Reproduce the fixed outer validation split

In [4]:
# Group identical prepared predictor rows, excluding the identifier and target.
group_ids = pd.util.hash_pandas_object(model_train.drop(columns=["id", "music_hit"]), index=False)
group_summary = pd.DataFrame({
    "group_id": group_ids.to_numpy(),
    "target": model_train["music_hit"].to_numpy(),
}).groupby("group_id", sort=False)["target"].agg(["count", "sum"])
# Stratify groups by their majority label; mixed-label ties are assigned to class 1.
group_labels = (2 * group_summary["sum"] >= group_summary["count"]).astype(int)
training_groups, validation_groups = train_test_split(
    group_summary.index.to_numpy(), test_size=0.2, random_state=RANDOM_STATE,
    stratify=group_labels.to_numpy(),
)
is_validation = group_ids.isin(validation_groups)
train_part = model_train.loc[~is_validation].copy()
valid_part = model_train.loc[is_validation].copy()
y_train = train_part["music_hit"].to_numpy()
y_valid = valid_part["music_hit"].to_numpy()
assert not set(training_groups).intersection(validation_groups)
assert not set(train_part["id"]).intersection(valid_part["id"])
assert len(train_part) + len(valid_part) == len(model_train)
assert set(np.unique(y_train)) == {0, 1} and set(np.unique(y_valid)) == {0, 1}
split_summary = pd.DataFrame([
    {"subset": name, "rows": len(frame), "hit proportion": frame["music_hit"].mean()}
    for name, frame in (("training", train_part), ("validation", valid_part))
])
display(split_summary)
validation_split = pd.DataFrame({
    "id": model_train["id"].to_numpy(),
    "subset": np.where(is_validation.to_numpy(), "validation", "training"),
    "group_id": group_ids.to_numpy(),
})

training_group_ids = group_ids.loc[~is_validation].to_numpy()


,subset,rows,hit proportion
0,training,280031,0.103389
1,validation,70018,0.103331


## 4. Define the controlled before/after encoding

We fit smoothed encodings on other training groups for each row. The validation set receives rates fitted only from outer-training labels. We standardize that one new column and keep the classifier and all other features unchanged.

In [5]:
NUMERIC_FEATURES = [
    "duration_sec", "release_year", "audio_dance", "audio_energy", "audio_loudness",
    "audio_speech", "audio_acoustic", "audio_instrumental", "audio_live",
    "audio_valence", "audio_tempo", "artist_followers_log1p",
]
CATEGORICAL_FEATURES = ["album_type", "audio_key", "time_signature"]
BINARY_FEATURES = ["audio_mode", "explicit_lyrics"]
INPUT_COLUMNS = NUMERIC_FEATURES + CATEGORICAL_FEATURES + BINARY_FEATURES
assert "id" not in INPUT_COLUMNS and "music_hit" not in INPUT_COLUMNS
assert "artist_followers" not in INPUT_COLUMNS

def fit_base_feature_map(frame):
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler()),
        ]), NUMERIC_FEATURES),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL_FEATURES),
        ("binary", "passthrough", BINARY_FEATURES),
    ], sparse_threshold=1.0)
    # Fit model-time transformations on the training partition only.
    matrix = csr_matrix(preprocessor.fit_transform(frame[INPUT_COLUMNS]), dtype=np.float64)
    genre_matrix = frame[GENRE_COLUMNS].sparse.to_coo().tocsr()
    matrix = hstack([matrix, genre_matrix], format="csr", dtype=np.float64)
    assert np.isfinite(matrix.data).all()
    return matrix, {
        "preprocessor": preprocessor, "input_columns": INPUT_COLUMNS.copy(),
        "genre_columns": GENRE_COLUMNS.copy(),
        "feature_names": preprocessor.get_feature_names_out().tolist() + GENRE_COLUMNS,
    }

def transform_base_features(frame, feature_map):
    matrix = csr_matrix(feature_map["preprocessor"].transform(frame[feature_map["input_columns"]]), dtype=np.float64)
    genre_matrix = frame[feature_map["genre_columns"]].sparse.to_coo().tocsr()
    matrix = hstack([matrix, genre_matrix], format="csr", dtype=np.float64)
    assert matrix.shape[1] == len(feature_map["preprocessor"].get_feature_names_out()) + len(feature_map["genre_columns"])
    assert np.isfinite(matrix.data).all()
    return matrix

def fit_record_label(frame, target, groups):
    labels = frame[["record_label"]]
    target = np.asarray(target)
    groups = np.asarray(groups)
    if len(labels) != len(target) or len(labels) != len(groups):
        raise ValueError("Labels, targets, and group IDs must have matching row counts.")
    if len(np.unique(groups)) < LABEL_FOLDS:
        raise ValueError("Target encoding requires at least five distinct training groups.")
    encoded = np.empty((len(frame), 1), dtype=float)
    seen = np.zeros(len(frame), dtype=int)
    splitter = GroupKFold(n_splits=LABEL_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    for fit_rows, held_out_rows in splitter.split(labels, target, groups):
        assert not set(groups[fit_rows]).intersection(groups[held_out_rows])
        fold_encoder = TargetEncoder(target_type="binary", smooth="auto")
        fold_encoder.fit(labels.iloc[fit_rows], target[fit_rows])
        encoded[held_out_rows] = fold_encoder.transform(labels.iloc[held_out_rows])
        seen[held_out_rows] += 1
    assert (seen == 1).all()
    assert np.isfinite(encoded).all() and ((encoded >= 0) & (encoded <= 1)).all()
    # This encoder is used only on validation, test, or future rows.
    encoder = TargetEncoder(target_type="binary", smooth="auto")
    encoder.fit(labels, target)
    return encoded, encoder


def fit_feature_map(frame, target, groups):
    matrix, feature_map = fit_base_feature_map(frame)
    encoded, encoder = fit_record_label(frame, target, groups)
    label_scaler = StandardScaler()
    scaled = label_scaler.fit_transform(encoded)
    matrix = hstack([matrix, csr_matrix(scaled)], format="csr", dtype=np.float64)
    feature_map.update({
        "record_label_encoder": encoder,
        "record_label_scaler": label_scaler,
        "target_encoded_columns": ["record_label"],
        "record_label_encoding": {"folds": LABEL_FOLDS, "group_aware": True, "smoothing": "auto"},
        "record_label_oof_range": [float(encoded.min()), float(encoded.max())],
    })
    feature_map["feature_names"] += ["record_label__target_mean"]
    assert matrix.shape[1] == len(feature_map["feature_names"])
    return matrix, feature_map


def transform_features(frame, feature_map):
    matrix = transform_base_features(frame, feature_map)
    encoded = feature_map["record_label_encoder"].transform(frame[["record_label"]])
    assert np.isfinite(encoded).all() and ((encoded >= 0) & (encoded <= 1)).all()
    scaled = feature_map["record_label_scaler"].transform(encoded)
    matrix = hstack([matrix, csr_matrix(scaled)], format="csr", dtype=np.float64)
    assert matrix.shape[1] == len(feature_map["feature_names"])
    return matrix


def evaluate_probabilities(y, probabilities):
    predictions = (probabilities >= THRESHOLD).astype(int)
    return {
        "roc_auc": roc_auc_score(y, probabilities),
        "average_precision": average_precision_score(y, probabilities),
        "accuracy": accuracy_score(y, predictions),
        "precision": precision_score(y, predictions, zero_division=0),
        "recall": recall_score(y, predictions, zero_division=0),
        "f1": f1_score(y, predictions, zero_division=0),
    }

def fit_logistic(matrix, target):
    model = LogisticRegression(C=1.0, solver="lbfgs", max_iter=MAX_ITER, tol=1e-4, random_state=RANDOM_STATE)
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always", ConvergenceWarning)
        with threadpool_limits(limits=2):
            model.fit(matrix, target)
    if any(issubclass(item.category, ConvergenceWarning) for item in captured):
        raise RuntimeError("Logistic regression did not converge; review the solver settings before using these scores.")
    assert model.classes_.tolist() == [0, 1]
    return model

## 5. Measure the contribution of record labels

In [6]:
comparison_rows = []
comparison_probabilities = {}
for with_label, name in ((False, "without_record_label"), (True, "with_record_label")):
    start = time.perf_counter()
    if with_label:
        X_train, feature_map = fit_feature_map(train_part, y_train, training_group_ids)
        X_valid = transform_features(valid_part, feature_map)
    else:
        X_train, feature_map = fit_base_feature_map(train_part)
        X_valid = transform_base_features(valid_part, feature_map)
    model = fit_logistic(X_train, y_train)
    with threadpool_limits(limits=2):
        probabilities = model.predict_proba(X_valid)[:, 1]
    comparison_probabilities[name] = probabilities
    comparison_rows.append({
        "configuration": name, **evaluate_probabilities(y_valid, probabilities),
        "features": X_train.shape[1], "fit_seconds": time.perf_counter() - start,
    })
    print(f"{name}: accuracy {comparison_rows[-1]['accuracy']:.2%}; ROC AUC {comparison_rows[-1]['roc_auc']:.5f}", flush=True)
    del X_train, X_valid
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(REPORT_DIR / "metrics.csv", index=False)
display(comparison.round(5))
label_encoder = feature_map["record_label_encoder"]
validation_rates = label_encoder.transform(valid_part[["record_label"]])
unseen_labels = ~valid_part["record_label"].isin(train_part["record_label"])
if unseen_labels.any():
    np.testing.assert_allclose(validation_rates[unseen_labels.to_numpy(), 0], y_train.mean())
assert ((validation_rates >= 0) & (validation_rates <= 1)).all()
print(f"Unseen validation-label rows: {int(unseen_labels.sum()):,}; fallback: {y_train.mean():.5f}")
print("Training cross-fitted rate range:", feature_map["record_label_oof_range"])


without_record_label: accuracy 92.15%; ROC AUC 0.91643
with_record_label: accuracy 93.16%; ROC AUC 0.94094
Unseen validation-label rows: 2,921; fallback: 0.10339
Training cross-fitted rate range: [0.0, 1.0]


,configuration,roc_auc,average_precision,accuracy,precision,recall,f1,features,fit_seconds
0,without_record_label,0.91643,0.62718,0.92152,0.68095,0.45252,0.54372,1035,3.06121
1,with_record_label,0.94094,0.70659,0.93156,0.71958,0.55328,0.62557,1036,4.11140


## 6. Record our decision

In [7]:
before = comparison.loc[comparison["configuration"].eq("without_record_label")].iloc[0]
after = comparison.loc[comparison["configuration"].eq("with_record_label")].iloc[0]
lines = [
    "# Record-label target encoding", "",
    "We studied the prepared `record_label` column and compared the existing logistic regression with the same model plus one encoded label feature. "
    "Both configurations used the same prepared data, outer validation split, model settings, and threshold of 0.5.", "",
    "## Dataset profile", "",
    "| Dataset | Distinct record labels | Missing | Unknown | Labels with one track |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for _, row in label_summary.iterrows():
    lines.append(f"| {row['dataset']} | {row['distinct record labels']} | {row['missing values']} | {row['unknown values']} | {row['labels with one track']} |")
lines += [
    "", "## Encoding", "",
    "We represented each label through a smoothed estimate of its hit rate. "
    "We cross-fitted training values using five internal folds with identical predictor groups kept together. "
    "We fitted an encoder only on the complete training partition for validation, with unseen categories receiving the training hit rate. "
    "We did not use validation targets to calculate encodings. "
    "The rates are in [0, 1] before a StandardScaler fitted on cross-fitted training values.", "",
    "## Validation comparison", "",
    "| Configuration | Accuracy | ROC AUC | Precision | Recall | F1 |",
    "| --- | ---: | ---: | ---: | ---: | ---: |",
]
for _, row in comparison.iterrows():
    lines.append(f"| {row['configuration']} | {row['accuracy']:.5f} | {row['roc_auc']:.5f} | {row['precision']:.5f} | {row['recall']:.5f} | {row['f1']:.5f} |")
lines += [
    "", "## Decision", "",
    f"We retained record-label target encoding after accuracy increased by {(after['accuracy']-before['accuracy'])*100:.3f} percentage points "
    f"and ROC AUC changed from {before['roc_auc']:.5f} to {after['roc_auc']:.5f} on this validation split. "
    "We integrated only the adopted configuration into notebooks 08 and delivery. "
    "We kept the supervised encoding inside model fitting and left raw and prepared CSV files unchanged. "
    "We saved the encoder and scaler with the classifier for inference.", "",
    "This comparison uses a reused development holdout, so it is exploratory evidence rather than an independent final performance estimate. "
    "The fixed genre columns were selected earlier during prepared-data construction.", "",
]
(REPORT_DIR / "summary.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Study report saved to {REPORT_DIR / 'summary.md'}")


Study report saved to /home/goncalotorcato/ist1109358/CD/Projeto/CD/reports/record_label_encoding/summary.md
